### 1. Simple Node Parser(Text Splitter)

계층 구조가 뚜렷하지 않은 일반 텍스트를 작은 조각(Node)으로 나누는 예제입니다.
Node는 이후 임베딩과 검색의 기본 단위가 되므로, 너무 작거나 너무 크게 자르지 않는 것이 중요합니다.

<!-- 학습 보강 셀 -->

## Text Splitter를 비교해서 봐야 하는 이유

같은 원문도 어떤 splitter를 쓰느냐에 따라 Node 경계가 달라집니다.
Node 경계가 검색 단위가 되기 때문에, splitter 선택은 RAG의 검색 품질과 직접 연결됩니다.

In [1]:
from llama_index.core.node_parser import TokenTextSplitter, SentenceSplitter

In [2]:
text = """
LlamaIndex는 대규모 언어 모델(LLM)을 사용하여 개인 데이터를 처리하는 데이터 프레임워크입니다. 이 프레임워크는 데이터 수집부터 처리, 검색까지 전체 과정을 지원합니다.

주요 구성 요소는 다음과 같습니다:
Documents는 원시 데이터를 표현하는 기본 단위입니다. 텍스트 파일, PDF, 웹페이지 등 다양한 소스의 데이터를 포함할 수 있습니다.
Nodes는 Documents를 더 작은 단위로 분할한 것으로, LLM이 효과적으로 처리할 수 있는 크기입니다.

데이터 처리 과정은 다음과 같습니다:
먼저 데이터 로더를 사용하여 Documents를 생성합니다. 그 다음 Node Parser를 통해 Documents를 Nodes로 분할합니다.
마지막으로 인덱스를 구축하여 효율적인 검색이 가능하도록 합니다.

LlamaIndex는 다양한 인덱스 유형을 제공합니다. VectorStoreIndex는 임베딩 기반 검색을, SummaryIndex는 요약 기반 검색을 지원합니다.
또한 하이브리드 검색, 재순위화 등 고급 검색 기능도 제공하여 검색 품질을 향상시킬 수 있습니다.
"""

<!-- 학습 보강 셀 -->

## 예제 원문을 먼저 읽어보기

분할 결과를 평가하려면 먼저 원문 구조를 알아야 합니다.
이 예제 텍스트는 개념 설명, 구성 요소, 처리 과정처럼 문단 구조가 있으므로 문장 기반 분할과 토큰 기반 분할의 차이를 비교하기 좋습니다.

In [3]:
# 1. TokenTextSplitter: 토큰 수 기준 분할
# - 문장 의미보다 토큰 길이를 우선하므로, 길이 제어가 중요한 경우에 유용합니다.
# - chunk_overlap은 앞 Node의 끝부분을 다음 Node에도 포함해 문맥 단절을 줄입니다.
token_splitter = TokenTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
)

token_nodes = token_splitter.split_text(text)

for i, node in enumerate(token_nodes, start=1):
    print()
    print(f'Token Node {i}:')
    print(node)


Token Node 1:
LlamaIndex는 대규모 언어 모델(LLM)을 사용하여 개인 데이터를 처리하는 데이터 프레임워크입니다. 이 프레임워크는 데이터 수집부터 처리, 검색까지 전체 과정을 지원합니다.

주요 구성 요소는 다음과 같습니다:
Documents는 원시 데이터를 표현하는 기본

Token Node 2:
같습니다:
Documents는 원시 데이터를 표현하는 기본 단위입니다. 텍스트 파일, PDF, 웹페이지 등 다양한 소스의 데이터를 포함할 수 있습니다.
Nodes는 Documents를 더 작은 단위로 분할한 것으로, LLM이 효과적으로 처리할 수 있는 크기입니다.

데이터 처리 과정은 다음과

Token Node 3:
처리할 수 있는 크기입니다.

데이터 처리 과정은 다음과 같습니다:
먼저 데이터 로더를 사용하여 Documents를 생성합니다. 그 다음 Node Parser를 통해 Documents를 Nodes로 분할합니다.
마지막으로 인덱스를 구축하여 효율적인 검색이 가능하도록 합니다.

LlamaIndex는 다양한

Token Node 4:
검색이 가능하도록 합니다.

LlamaIndex는 다양한 인덱스 유형을 제공합니다. VectorStoreIndex는 임베딩 기반 검색을, SummaryIndex는 요약 기반 검색을 지원합니다.
또한 하이브리드 검색, 재순위화 등 고급 검색 기능도 제공하여 검색

Token Node 5:
등 고급 검색 기능도 제공하여 검색 품질을 향상시킬 수 있습니다.


<!-- 학습 보강 셀 -->

## TokenTextSplitter 결과 해석

토큰 기준 분할은 길이를 안정적으로 제어할 수 있지만, 문장 중간에서 끊길 수 있습니다.
모델 입력 길이 제한을 엄격히 맞춰야 하는 경우에는 유리하지만, 사람이 읽는 문맥은 다소 어색해질 수 있습니다.

In [4]:
# 2. SentenceSplitter: 문장 경계 기준 분할
# - 가능한 한 문장을 깨지 않고 나누므로 RAG 검색 품질에 더 유리한 경우가 많습니다.
sentence_splitter = SentenceSplitter(
    chunk_size=100,
    chunk_overlap=20,
)

sentence_nodes = sentence_splitter.split_text(text)

for i, node in enumerate(sentence_nodes, start=1):
    print()
    print(f'Sentence Node {i}:')
    print(node)


Sentence Node 1:
LlamaIndex는 대규모 언어 모델(LLM)을 사용하여 개인 데이터를 처리하는 데이터 프레임워크입니다. 이 프레임워크는 데이터 수집부터 처리, 검색까지 전체 과정을 지원합니다.

Sentence Node 2:
주요 구성 요소는 다음과 같습니다:
Documents는 원시 데이터를 표현하는 기본 단위입니다. 텍스트 파일, PDF, 웹페이지 등 다양한 소스의 데이터를 포함할 수 있습니다.

Sentence Node 3:
Nodes는 Documents를 더 작은 단위로 분할한 것으로, LLM이 효과적으로 처리할 수 있는 크기입니다.

데이터 처리 과정은 다음과 같습니다:
먼저 데이터 로더를 사용하여 Documents를 생성합니다. 그 다음 Node Parser를 통해 Documents를 Nodes로 분할합니다.

Sentence Node 4:
그 다음 Node Parser를 통해 Documents를 Nodes로 분할합니다.
마지막으로 인덱스를 구축하여 효율적인 검색이 가능하도록 합니다.

LlamaIndex는 다양한 인덱스 유형을 제공합니다.

Sentence Node 5:
VectorStoreIndex는 임베딩 기반 검색을, SummaryIndex는 요약 기반 검색을 지원합니다.
또한 하이브리드 검색, 재순위화 등 고급 검색 기능도 제공하여 검색 품질을 향상시킬 수 있습니다.


<!-- 학습 보강 셀 -->

## SentenceSplitter 결과 해석

문장 기준 분할은 의미 단위가 더 자연스럽게 유지되는 편입니다.
질문이 문장이나 문단의 의미를 묻는 RAG에서는 일반적으로 더 이해하기 쉬운 검색 결과를 제공합니다.

#### Structured Node Parser

문서의 구조를 유지하면서 의미 있는 블록으로 분할하는 방식입니다.
Markdown, JSON, HTML처럼 제목/태그/키 구조가 있는 문서에서 특히 유용합니다.

<!-- 학습 보강 셀 -->

## 구조화 문서는 전용 Parser가 유리한 이유

Markdown, HTML, JSON은 단순 문자열이 아니라 제목, 태그, 키-값 같은 구조를 갖습니다.
전용 Parser를 쓰면 이 구조를 검색 단위에 반영할 수 있어, 섹션별 검색이나 계층적 이해에 더 적합합니다.

In [5]:
# JSONNodeParser
# - JSON의 계층 구조를 고려해 노드를 생성합니다.
from llama_index.core.node_parser import JSONNodeParser
from llama_index.core import SimpleDirectoryReader

json_docs = SimpleDirectoryReader(
    input_dir='../NewData',
    required_exts=['.json'],
).load_data()

parser = JSONNodeParser()
json_nodes = parser.get_nodes_from_documents(json_docs)

print('JSON 노드 개수:', len(json_nodes))

JSON 노드 개수: 2


In [6]:
# HTMLNodeParser
# - HTML 태그 구조를 기준으로 본문을 의미 단위로 나눕니다.
from llama_index.core.node_parser import HTMLNodeParser
from llama_index.core import SimpleDirectoryReader

html_docs = SimpleDirectoryReader(
    input_dir='../NewData',
    required_exts=['.html'],
).load_data()

parser = HTMLNodeParser()
html_nodes = parser.get_nodes_from_documents(html_docs)

print('HTML 노드 개수:', len(html_nodes))

HTML 노드 개수: 4


In [7]:
# MarkdownNodeParser
# - 제목(#, ## 등)과 섹션 구조를 유지해 노드를 만듭니다.
from llama_index.core.node_parser import MarkdownNodeParser
from llama_index.core import SimpleDirectoryReader

md_docs = SimpleDirectoryReader(
    input_dir='../NewData',
    required_exts=['.md'],
).load_data()

parser = MarkdownNodeParser()
md_nodes = parser.get_nodes_from_documents(md_docs)

print('Markdown 노드 개수:', len(md_nodes))

Markdown 노드 개수: 3
